In [ ]:
import json
import os
from PIL import Image
from underthesea import text_normalize, pos_tag
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Đường dẫn
json_path = "/mnt/VLAI_data/ViVQA-X/ViVQA-X_val.json"
coco_img_dir = "/mnt/VLAI_data/COCO_Images/val2014/"

# Đọc file JSON
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Hàm trích xuất danh từ
def extract_nouns(text):
    normalized = text_normalize(text)
    tagged = pos_tag(normalized)
    nouns = [word for word, tag in tagged if tag in ['N', 'Np']]
    return nouns

# Xử lý tất cả samples
samples_with_nouns = []
for item in data:
    img_path = os.path.join(coco_img_dir, item["image_name"])
    image = Image.open(img_path).convert("RGB")
    
    # Trích xuất danh từ từ câu hỏi
    question_nouns = extract_nouns(item["question"])
    
    # Trích xuất danh từ từ giải thích
    explanation_nouns = []
    for exp in item["explanation"]:
        explanation_nouns.extend(extract_nouns(exp))
    
    sample = {
        "question": item["question"],
        "question_nouns": question_nouns,
        "image": image,
        "image_path": img_path,
        "explanation": item["explanation"],
        "explanation_nouns": list(set(explanation_nouns)),  # unique nouns
        "answer": item["answer"],
        "question_id": item["question_id"]
    }
    samples_with_nouns.append(sample)

# Kiểm tra kết quả
print("Câu hỏi:", samples_with_nouns[0]["question"])
print("Danh từ trong câu hỏi:", samples_with_nouns[0]["question_nouns"])
print("\nGiải thích:", samples_with_nouns[0]["explanation"])
print("Danh từ trong giải thích:", samples_with_nouns[0]["explanation_nouns"])


Câu hỏi: Đây có phải là bức ảnh chụp nhiều độ phơi sáng của vận động viên trượt tuyết mặc áo đen không?
Danh từ trong câu hỏi: ['ảnh', 'độ', 'vận động viên', 'trượt tuyết', 'áo']

Giải thích: ['hình người mặc cùng một bộ quần áo khi trượt xuống dốc', 'có vẻ như là cùng một người trượt tuyết làm những pha nhào lộn khác nhau', 'cùng một người trượt tuyết xuất hiện nhiều lần']
Danh từ trong giải thích: ['bộ quần áo', 'dốc', 'khi', 'lần', 'người', 'pha nhào lộn', 'nhau', 'hình']
